## Establish Connection

In [1]:
# Import Libraries
import pandas as pd
import mysql.connector as connector

In [2]:
# Load All CSV files
df_provider = pd.read_csv("C:/Users/Nil/Documents/GUVI PROJECT/Food Waste Management/data/providers_data.csv")
df_receiver = pd.read_csv("C:/Users/Nil/Documents/GUVI PROJECT/Food Waste Management/data/receivers_data.csv")
df_food = pd.read_csv("C:/Users/Nil/Documents/GUVI PROJECT/Food Waste Management/data/food_listings_data.csv")
df_claim = pd.read_csv("C:/Users/Nil/Documents/GUVI PROJECT/Food Waste Management/data/claims_data.csv")

In [3]:
# Basic Information about DataFrames
tables = [df_provider, df_receiver, df_food, df_claim]

for table in tables:
    table.info()
    print("\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Provider_ID  1000 non-null   int64 
 1   Name         1000 non-null   object
 2   Type         1000 non-null   object
 3   Address      1000 non-null   object
 4   City         1000 non-null   object
 5   Contact      1000 non-null   object
dtypes: int64(1), object(5)
memory usage: 47.0+ KB


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Receiver_ID  1000 non-null   int64 
 1   Name         1000 non-null   object
 2   Type         1000 non-null   object
 3   City         1000 non-null   object
 4   Contact      1000 non-null   object
dtypes: int64(1), object(4)
memory usage: 39.2+ KB


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 

In [4]:
#  Convert Expiry_Date in food_listings_data
df_food['Expiry_Date'] = pd.to_datetime(df_food['Expiry_Date'])

#  Convert Timestamp in claims_data
df_claim['Timestamp'] = pd.to_datetime(df_claim['Timestamp'])

In [5]:
# Check Null Values
tables = [df_provider, df_receiver, df_food, df_claim]

for table in tables:
    print(table.isnull().sum().any())

False
False
False
False


In [6]:
# Check Duplicate Values
tables = [df_provider, df_receiver, df_food, df_claim]

for table in tables:
    print(table.duplicated().sum())

0
0
0
0


## Establish Connection

In [7]:
connection = connector.connect(
    host="127.0.0.1",
    user="root",
    password="1997"
)
cursor = connection.cursor()

## Create Database and Tables

In [8]:
# Database Creation
cursor.execute("""CREATE DATABASE IF NOT EXISTS food_waste_management""")
cursor.execute("""USE food_waste_management""")

In [9]:
# Create (providers_data, receivers_data, food_data, claim_data) Table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS providers_data (
        Provider_ID INT PRIMARY KEY,
        Name VARCHAR(255),
        Type VARCHAR(100),
        Address TEXT,
        City VARCHAR(100),
        Contact VARCHAR(50))
              """)

cursor.execute("""
    CREATE TABLE IF NOT EXISTS receivers_data (
        Receiver_ID INT NOT NULL,
        Name VARCHAR(255),
        Type VARCHAR(100),
        City VARCHAR(100),
        Contact VARCHAR(50))
              """)

cursor.execute("""
    CREATE TABLE IF NOT EXISTS food_listings_data (
        Food_ID INT NOT NULL,
        Food_Name VARCHAR(50), 
        Quantity INT,
        Expiry_Date DATETIME, 
        Provider_ID INT, 
        Provider_Type VARCHAR(50), 
        Location VARCHAR(50), 
        Food_Type VARCHAR(50), 
        Meal_Type VARCHAR(50))
              """)

cursor.execute("""
    CREATE TABLE IF NOT EXISTS claims_Data (
        Claim_ID INT,
        Food_ID INT,
        Receiver_ID INT, 
        Status VARCHAR(50), 
        Timestamp DATETIME)
              """)
connection.commit()

## Insert Data

In [10]:
# Providers_Data
for index, row in df_provider.iterrows():
    cursor.execute("""
        INSERT INTO providers_data (Provider_ID, Name, Type, Address, City, Contact) 
        VALUES (%s, %s, %s, %s, %s, %s)
    """, tuple(row)) 

# Receivers_Data
for index, row in df_receiver.iterrows():
    cursor.execute("""
        INSERT INTO receivers_data (Receiver_ID, Name, Type, City, Contact) 
        VALUES (%s, %s, %s, %s, %s)
    """, tuple(row)) 

# Food_Listings_Data
for index, row in df_food.iterrows():
    cursor.execute("""
        INSERT INTO food_listings_data (Food_ID, Food_Name, Quantity, Expiry_Date, Provider_ID, Provider_Type, Location, Food_Type, Meal_Type) 
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, tuple(row))

# Claims_Data
for index, row in df_claim.iterrows():
    cursor.execute("""
        INSERT INTO claims_data (Claim_ID, Food_ID, Receiver_ID, Status, timestamp) 
        VALUES (%s, %s, %s, %s, %s)
    """, tuple(row))

connection.commit()